# 25 — Context-Aware SQR Calibration

Consolidates legacy experiments 83 through 88.

1. **SQR gate decomposition** — decompose unitary into SQR gate sequence (Exp 83)
2. **SQR closed-loop calibration** — hardware calibration of SQR gates (Exp 84)
3. **SQR process tomography** — full process matrix characterization (Exp 85)
4. **SQR fidelity benchmarking** — gate set tomography / RB with SQR gates (Exp 86)
5. **SQR context-aware update** — context-dependent parameter corrections (Exp 87)
6. **SQR final validation** — end-to-end validation run (Exp 88)

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    CalibrationOrchestrator,
    load_stage_checkpoint,
    open_notebook_stage,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="25_context_aware_sqr_calibration",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

prev_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="24_free_evolution_tomography",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if prev_checkpoint is not None:
    print(f"Prior stage 24 status: {prev_checkpoint['status']}")

## 2. SQR Calibration Defaults

In [ ]:
SQR_N_AVG = 10000
SQR_MAX_RANK = 3
SQR_DURATION_NS = 1000

print("SQR calibration settings:")
print(f"  n_avg: {SQR_N_AVG}")
print(f"  max_rank: {SQR_MAX_RANK}")
print(f"  duration: {SQR_DURATION_NS} ns")

## 3. SQR Gate Decomposition — Exp 83

Decompose target unitary into SQR gate sequences.

In [ ]:
orchestrator = CalibrationOrchestrator(session)

decomposition_result = orchestrator.decompose_unitary(
    max_rank=SQR_MAX_RANK,
    duration_ns=SQR_DURATION_NS,
)

print("SQR decomposition complete.")
print(f"  Number of SQR gates: {decomposition_result.get('n_gates', 'N/A')}")

## 4. SQR Closed-Loop Calibration — Exp 84

Run closed-loop hardware calibration of SQR gate parameters.

In [ ]:
sqr_cal_result = orchestrator.run_closed_loop_calibration(
    n_avg=SQR_N_AVG,
)

print("SQR closed-loop calibration complete.")

## 5. SQR Process Tomography — Exp 85

Full process tomography of the calibrated SQR gate.

In [ ]:
sqr_tomo_result = orchestrator.run_process_tomography(
    n_avg=SQR_N_AVG,
)

print("SQR process tomography complete.")

## 6. SQR Fidelity Benchmarking — Exp 86

Gate set tomography / RB with SQR gates.

In [ ]:
sqr_fidelity_result = orchestrator.run_fidelity_benchmarking(
    n_avg=SQR_N_AVG,
)

print("SQR fidelity benchmarking complete.")

## 7. SQR Context-Aware Update — Exp 87

Apply context-dependent corrections to SQR parameters.

In [ ]:
sqr_context_result = orchestrator.apply_context_corrections(
    n_avg=SQR_N_AVG,
)

print("SQR context-aware update complete.")

## 8. SQR Final Validation — Exp 88

End-to-end validation of calibrated SQR gate set.

In [ ]:
sqr_validation_result = orchestrator.run_final_validation(
    n_avg=SQR_N_AVG,
)

print("SQR final validation complete.")

## 9. Save Checkpoint

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="25_context_aware_sqr_calibration",
    status="calibrated",
    summary="Full SQR calibration pipeline: decomposition, closed-loop cal, process tomo, benchmarking, context corrections, validation.",
    consumed_inputs={
        "sqr_n_avg": SQR_N_AVG,
        "sqr_max_rank": SQR_MAX_RANK,
        "sqr_duration_ns": SQR_DURATION_NS,
    },
    persisted_outputs={},
    advisory_outputs={},
    next_stage="26_sequential_simulation",
    notes=[
        "This notebook drives the CalibrationOrchestrator end-to-end.",
        "Context-aware corrections are essential for multi-gate sequence fidelity.",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")